# 🎵 Feature Extraction Pipeline – VCTK Corpus

**Input:** `data/processed/<speaker>/<file>.wav` (16kHz, 5s, 500 files)  
**Output:**
- `data/features/<speaker>/<file>.npy` — MFCC thô `(40, 157)` dùng cho DTW
- `records` — list metadata + embedding `(99,)` cho insert PostgreSQL

**Vector 99 chiều:**
```
[0:40]   MFCC mean          (40 chiều)
[40:80]  MFCC std           (40 chiều)
[80:87]  Spectral Contrast  ( 7 chiều)
[87:99]  Chroma STFT        (12 chiều)
```

## Cell 0 – Cấu hình toàn cục

In [1]:
from pathlib import Path

# ─── Root project (Beta/) ────────────────────────────────────────────────────
# Notebook nằm tại src/feature_extraction/ → lên 2 cấp
PROJECT_ROOT  = Path().resolve().parents[1]   # Beta/

# ─── Đường dẫn dữ liệu ───────────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"   # input: .wav đã tiền xử lý
FEATURES_DIR  = PROJECT_ROOT / "data" / "features"    # output: .npy đặc trưng thô

# ─── Tham số MFCC ────────────────────────────────────────────────────────────
TARGET_SR  = 16_000   # Hz – khớp với tiền xử lý
N_MFCC     = 40       # Số hệ số MFCC giữ lại
N_FFT      = 2048     # Kích thước cửa sổ FFT (128ms tại 16kHz)
HOP_LENGTH = 512      # Bước nhảy giữa frames (32ms tại 16kHz, 75% overlap)
N_MELS     = 128      # Số Mel filter banks

# ─── Tham số Spectral Contrast ───────────────────────────────────────────────
SC_N_BANDS = 6        # Số dải tần → 7 hệ số kể cả dải bass
SC_FMIN    = 200.0    # Hz – tần số bắt đầu dải đầu tiên

# ─── Chroma STFT dùng chung N_FFT và HOP_LENGTH ─────────────────────────────

# ─── Tổng chiều embedding ────────────────────────────────────────────────────
EMBEDDING_DIM = N_MFCC * 2 + (SC_N_BANDS + 1) + 12   # = 40+40+7+12 = 99

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FEATURES_DIR  : {FEATURES_DIR}")
print(f"EMBEDDING_DIM : {EMBEDDING_DIM}")
assert EMBEDDING_DIM == 99, f"Kiểm tra lại công thức! Nhận được {EMBEDDING_DIM}"

PROJECT_ROOT  : C:\BTL_CSDLDPT\Beta
PROCESSED_DIR : C:\BTL_CSDLDPT\Beta\data\processed
FEATURES_DIR  : C:\BTL_CSDLDPT\Beta\data\features
EMBEDDING_DIM : 99


## Cell 1 – Import thư viện

In [2]:
import json
import logging
from collections import Counter

import numpy as np
import librosa
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print(f"librosa version : {librosa.__version__}")
print(f"numpy version   : {np.__version__}")

librosa version : 0.11.0
numpy version   : 1.26.4


## Cell 2 – Hàm trích xuất đặc trưng

### Sơ đồ luồng xử lý một file:
```
wav_path (16kHz, 5s)
    │
    ├─ librosa.feature.mfcc(n_mfcc=40, n_fft=2048, hop_length=512, n_mels=128)
    │   → shape (40, 157)
    │   ├─ mean(axis=1) → mfcc_mean (40,)
    │   └─ std(axis=1)  → mfcc_std  (40,)
    │
    ├─ librosa.feature.spectral_contrast(n_bands=6, fmin=200)
    │   → shape (7, 157)
    │   └─ mean(axis=1) → sc_mean (7,)
    │
    └─ librosa.feature.chroma_stft(n_fft=2048, hop_length=512)
        → shape (12, 157)
        └─ mean(axis=1) → chroma_mean (12,)
```

In [3]:
def extract_features(wav_path: Path) -> dict:
    """
    Trích xuất MFCC, Spectral Contrast, Chroma STFT từ một file .wav.

    Parameters
    ----------
    wav_path : Path
        Đường dẫn đến file .wav (đã tiền xử lý, 16kHz, 5 giây).

    Returns
    -------
    dict với các keys:
        'mfcc_mean'   : np.ndarray (40,)  float32 – MFCC trung bình theo thời gian
        'mfcc_std'    : np.ndarray (40,)  float32 – MFCC độ lệch chuẩn
        'sc_mean'     : np.ndarray (7,)   float32 – Spectral Contrast trung bình
        'chroma_mean' : np.ndarray (12,)  float32 – Chroma STFT trung bình
        'mfcc_raw'    : np.ndarray (40,157) float32 – MFCC thô (lưu .npy, dùng DTW)
    """
    # ── 1. Load audio ────────────────────────────────────────────────────────
    # sr=TARGET_SR → librosa resample nếu cần (file đã 16kHz → không đổi)
    # mono=True    → đảm bảo 1 kênh
    y, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)
    # y.shape = (80000,), y.dtype = float32, y ∈ [-1.0, 1.0]

    # ── 2. MFCC ──────────────────────────────────────────────────────────────
    # Pipeline: Pre-emphasis → Framing → Hamming window → FFT → Mel → Log → DCT
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,         # 40 hệ số DCT giữ lại
        n_fft=N_FFT,           # 2048 = cửa sổ 128ms
        hop_length=HOP_LENGTH, # 512 = bước nhảy 32ms, overlap 75%
        n_mels=N_MELS,         # 128 Mel filter banks
    )
    # mfcc.shape = (40, 157)  ← (n_mfcc, n_frames)

    mfcc_mean = np.mean(mfcc, axis=1).astype(np.float32)  # (40,) – hướng giọng
    mfcc_std  = np.std(mfcc,  axis=1).astype(np.float32)  # (40,) – biến động giọng

    # ── 3. Spectral Contrast ─────────────────────────────────────────────────
    # Pipeline: STFT → chia 6 sub-bands + 1 bass → peak - valley (dB) mỗi band
    sc = librosa.feature.spectral_contrast(
        y=y,
        sr=sr,
        n_bands=SC_N_BANDS,    # 6 dải → 7 hệ số (kể cả bass band)
        fmin=SC_FMIN,          # Bắt đầu từ 200 Hz
    )
    # sc.shape = (7, 157)  ← (n_bands+1, n_frames)

    sc_mean = np.mean(sc, axis=1).astype(np.float32)  # (7,) – tương phản phổ TB

    # ── 4. Chroma STFT ───────────────────────────────────────────────────────
    # Pipeline: STFT → ánh xạ tần số → 12 pitch classes (C,C#,...,B) → tổng hợp
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr,
        n_fft=N_FFT,           # Cùng cửa sổ với MFCC
        hop_length=HOP_LENGTH, # Cùng bước nhảy với MFCC
    )
    # chroma.shape = (12, 157)  ← (12 pitch classes, n_frames)

    chroma_mean = np.mean(chroma, axis=1).astype(np.float32)  # (12,) – phân bố pitch

    return {
        "mfcc_mean"   : mfcc_mean,    # (40,)     → embedding [0:40]
        "mfcc_std"    : mfcc_std,     # (40,)     → embedding [40:80]
        "sc_mean"     : sc_mean,      # (7,)      → embedding [80:87]
        "chroma_mean" : chroma_mean,  # (12,)     → embedding [87:99]
        "mfcc_raw"    : mfcc,         # (40, 157) → lưu vào .npy cho DTW
    }

## Cell 3 – Hàm xây dựng vector embedding

In [4]:
def build_embedding(features: dict) -> np.ndarray:
    """
    Ghép nối (concatenate) tất cả đặc trưng thành vector 99 chiều.

    Cấu trúc vector:
        [0:40]  MFCC mean          (40 chiều)
        [40:80] MFCC std           (40 chiều)
        [80:87] Spectral Contrast  ( 7 chiều)
        [87:99] Chroma STFT        (12 chiều)

    Returns
    -------
    np.ndarray shape (99,) dtype=float32
    """
    embedding = np.concatenate([
        features["mfcc_mean"],    # [0:40]
        features["mfcc_std"],     # [40:80]
        features["sc_mean"],      # [80:87]
        features["chroma_mean"],  # [87:99]
    ])  # shape: (99,)

    # Kiểm tra shape – assert lỗi sớm nếu tham số bị thay đổi
    assert embedding.shape == (EMBEDDING_DIM,), (
        f"Embedding shape không đúng: {embedding.shape}, expected ({EMBEDDING_DIM},). "
        f"Kiểm tra lại N_MFCC={N_MFCC}, SC_N_BANDS={SC_N_BANDS}"
    )
    return embedding  # (99,) float32

## Cell 4 – Test nhanh với 1 file

In [5]:
# Lấy file đầu tiên để test
test_files = sorted(PROCESSED_DIR.rglob("*.wav"))

if not test_files:
    print("❌ Không tìm thấy file .wav trong", PROCESSED_DIR)
    print("   Hãy chạy preprocess.ipynb trước!")
else:
    test_file = test_files[0]
    print(f"Test với file: {test_file.name}")
    print(f"Speaker: {test_file.parent.name}")
    print()

    # Trích xuất
    feats = extract_features(test_file)

    print("── Shapes đặc trưng ──")
    print(f"  mfcc_raw    : {feats['mfcc_raw'].shape}   (n_mfcc × n_frames)")
    print(f"  mfcc_mean   : {feats['mfcc_mean'].shape}")
    print(f"  mfcc_std    : {feats['mfcc_std'].shape}")
    print(f"  sc_mean     : {feats['sc_mean'].shape}")
    print(f"  chroma_mean : {feats['chroma_mean'].shape}")
    print()

    # Build embedding
    emb = build_embedding(feats)
    print("── Embedding ──")
    print(f"  Shape : {emb.shape}")
    print(f"  Dtype : {emb.dtype}")
    print(f"  Min   : {emb.min():.4f}")
    print(f"  Max   : {emb.max():.4f}")
    print(f"  Mean  : {emb.mean():.4f}")
    print()
    print("  MFCC mean (5 đầu) :", emb[0:5].round(2))
    print("  MFCC std  (5 đầu) :", emb[40:45].round(2))
    print("  Spectral Contrast :", emb[80:87].round(2))
    print("  Chroma STFT       :", emb[87:99].round(2))
    print()
    print("✅ Test passed! Embedding shape đúng (99,)")

Test với file: p225_010.wav
Speaker: p225

── Shapes đặc trưng ──
  mfcc_raw    : (40, 157)   (n_mfcc × n_frames)
  mfcc_mean   : (40,)
  mfcc_std    : (40,)
  sc_mean     : (7,)
  chroma_mean : (12,)

── Embedding ──
  Shape : (99,)
  Dtype : float32
  Min   : -394.1877
  Max   : 157.7948
  Mean  : 4.3825

  MFCC mean (5 đầu) : [-394.19   48.4     1.06   14.81    2.02]
  MFCC std  (5 đầu) : [157.79  49.81  23.77  17.71  15.84]
  Spectral Contrast : [22.02 15.99 18.74 17.52 17.57 18.84 23.81]
  Chroma STFT       : [0.48 0.44 0.34 0.28 0.29 0.29 0.32 0.31 0.33 0.37 0.4  0.48]

✅ Test passed! Embedding shape đúng (99,)


## Cell 5 – Xử lý toàn bộ 500 file

In [6]:
def process_all(processed_dir: Path, features_dir: Path) -> list:
    """
    Duyệt toàn bộ data/processed/, trích xuất đặc trưng,
    lưu .npy cho DTW, trả về records cho DB insert.

    Idempotent: Bỏ qua file đã xử lý (npy_path.exists()).

    Parameters
    ----------
    processed_dir : Path   – data/processed/
    features_dir  : Path   – data/features/

    Returns
    -------
    list[dict] – mỗi phần tử chứa metadata + embedding list[float]
    """
    records = []
    errors  = []

    # Duyệt đệ quy tất cả .wav trong processed/
    wav_files = sorted(processed_dir.rglob("*.wav"))
    log.info(f"Tìm thấy {len(wav_files)} file .wav để xử lý")

    for wav_path in tqdm(wav_files, desc="Trích xuất đặc trưng"):
        speaker = wav_path.parent.name          # 'p225'

        # ── Đường dẫn file .npy output ───────────────────────────────────────
        npy_dir  = features_dir / speaker       # data/features/p225/
        npy_dir.mkdir(parents=True, exist_ok=True)
        npy_path = npy_dir / wav_path.with_suffix(".npy").name
        # VD: data/features/p225/p225_001.npy

        try:
            # ── Trích xuất đặc trưng ─────────────────────────────────────────
            features  = extract_features(wav_path)
            embedding = build_embedding(features)

            # ── Lưu .npy (MFCC thô shape (40,157) cho DTW) ──────────────────
            if not npy_path.exists():
                np.save(str(npy_path), features["mfcc_raw"])
                # Đọc lại: np.load(npy_path) → (40, 157)

            # ── Ghi record ───────────────────────────────────────────────────
            records.append({
                "speaker"     : speaker,
                # Đường dẫn tương đối so với data/
                "file_path"   : str(wav_path.relative_to(processed_dir.parent)),
                "npy_path"    : str(npy_path.relative_to(features_dir.parent)),
                "sample_rate" : TARGET_SR,
                "duration_sec": 5.0,        # đã chuẩn hóa ở preprocess
                # embedding là list[float] → psycopg2 / JSON serializable
                "embedding"   : embedding.tolist(),
            })

        except Exception as e:
            log.warning(f"Lỗi khi xử lý {wav_path.name}: {e}")
            errors.append({"file": str(wav_path), "error": str(e)})

    log.info(f"Hoàn thành: {len(records)} file thành công, {len(errors)} lỗi")
    return records, errors


# ── Chạy pipeline ────────────────────────────────────────────────────────────
records, errors = process_all(PROCESSED_DIR, FEATURES_DIR)

20:50:33 [INFO] Tìm thấy 1012 file .wav để xử lý


Trích xuất đặc trưng:   0%|          | 0/1012 [00:00<?, ?it/s]

c:\Users\kaita\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
20:51:05 [INFO] Hoàn thành: 1012 file thành công, 0 lỗi


## Cell 6 – Báo cáo tổng kết

In [7]:
from collections import Counter

# Đếm số file mỗi speaker
speaker_counts = Counter(r["speaker"] for r in records)

# Đếm số file .npy đã lưu thực tế
npy_files = list(FEATURES_DIR.rglob("*.npy"))

print("=" * 60)
print("  TỔNG KẾT TRÍCH XUẤT ĐẶC TRƯNG")
print("=" * 60)
print(f"  Tổng file xử lý      : {len(records)}")
print(f"  File lỗi             : {len(errors)}")
print(f"  Số speakers          : {len(speaker_counts)}")
if speaker_counts:
    vals = list(speaker_counts.values())
    print(f"  File/speaker (min)   : {min(vals)}")
    print(f"  File/speaker (max)   : {max(vals)}")
    print(f"  File/speaker (avg)   : {len(records)/len(speaker_counts):.1f}")
print(f"  File .npy đã lưu     : {len(npy_files)}")
print()
print("── Cấu trúc vector embedding ──")
print(f"  Tổng chiều           : {EMBEDDING_DIM}")
print(f"  [0:40]   MFCC mean   : {N_MFCC} chiều")
print(f"  [40:80]  MFCC std    : {N_MFCC} chiều")
print(f"  [80:87]  Spectral Contrast : {SC_N_BANDS+1} chiều")
print(f"  [87:99]  Chroma STFT : 12 chiều")
print()
print("── Tham số trích xuất ──")
print(f"  Sample rate          : {TARGET_SR} Hz")
print(f"  n_fft                : {N_FFT} ({N_FFT/TARGET_SR*1000:.0f}ms/frame)")
print(f"  hop_length           : {HOP_LENGTH} ({HOP_LENGTH/TARGET_SR*1000:.0f}ms/step)")
print(f"  n_mels               : {N_MELS}")
print(f"  n_mfcc               : {N_MFCC}")
print(f"  Spectral Contrast fmin: {SC_FMIN} Hz, n_bands: {SC_N_BANDS}")
print("=" * 60)

if records:
    print(f"\n✅ Trích xuất đặc trưng hoàn tất! Sẵn sàng insert vào PostgreSQL.")
else:
    print("\n❌ Không có records. Kiểm tra lại PROCESSED_DIR và log.")

if errors:
    print(f"\n⚠️  {len(errors)} file lỗi:")
    for e in errors[:5]:
        print(f"   - {e['file']}: {e['error']}")

  TỔNG KẾT TRÍCH XUẤT ĐẶC TRƯNG
  Tổng file xử lý      : 1012
  File lỗi             : 0
  Số speakers          : 61
  File/speaker (min)   : 16
  File/speaker (max)   : 20
  File/speaker (avg)   : 16.6
  File .npy đã lưu     : 1012

── Cấu trúc vector embedding ──
  Tổng chiều           : 99
  [0:40]   MFCC mean   : 40 chiều
  [40:80]  MFCC std    : 40 chiều
  [80:87]  Spectral Contrast : 7 chiều
  [87:99]  Chroma STFT : 12 chiều

── Tham số trích xuất ──
  Sample rate          : 16000 Hz
  n_fft                : 2048 (128ms/frame)
  hop_length           : 512 (32ms/step)
  n_mels               : 128
  n_mfcc               : 40
  Spectral Contrast fmin: 200.0 Hz, n_bands: 6

✅ Trích xuất đặc trưng hoàn tất! Sẵn sàng insert vào PostgreSQL.


## Cell 7 – Lưu records ra JSON (backup trước khi insert DB)

In [8]:
# Lưu records ra JSON để backup và kiểm tra trước khi insert DB
# Lưu ý: embedding là list[float] → JSON serializable

records_json_path = PROJECT_ROOT / "data" / "feature_records.json"

with open(str(records_json_path), "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"✅ Đã lưu {len(records)} records → {records_json_path}")
print(f"   Dung lượng: {records_json_path.stat().st_size / 1024:.1f} KB")

# Preview một record
if records:
    sample = records[0].copy()
    sample["embedding"] = sample["embedding"][:5] + ["..."]  # rút gọn cho display
    print("\n── Preview record đầu tiên ──")
    for k, v in sample.items():
        print(f"  {k:15s}: {v}")

✅ Đã lưu 1012 records → C:\BTL_CSDLDPT\Beta\data\feature_records.json
   Dung lượng: 2815.5 KB

── Preview record đầu tiên ──
  speaker        : p225
  file_path      : processed\p225\p225_010.wav
  npy_path       : features\p225\p225_010.npy
  sample_rate    : 16000
  duration_sec   : 5.0
  embedding      : [-394.1876525878906, 48.398380279541016, 1.0568090677261353, 14.809479713439941, 2.0154037475585938, '...']


## Cell 8 – Kiểm tra chất lượng embedding (Sanity check)

In [9]:
if len(records) >= 2:
    # Cosine similarity giữa 2 file cùng speaker
    same_speaker = [r for r in records if r["speaker"] == records[0]["speaker"]]
    diff_speaker = [r for r in records if r["speaker"] != records[0]["speaker"]]

    def cosine_similarity(a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    emb0 = records[0]["embedding"]

    # Cùng speaker
    if len(same_speaker) >= 2:
        sim_same = cosine_similarity(emb0, same_speaker[1]["embedding"])
        print(f"✅ Cosine similarity (cùng speaker {records[0]['speaker']}): {sim_same:.4f}")

    # Khác speaker
    if diff_speaker:
        sim_diff = cosine_similarity(emb0, diff_speaker[0]["embedding"])
        print(f"   Cosine similarity (khác speaker {diff_speaker[0]['speaker']}): {sim_diff:.4f}")
        print()
        if sim_same > sim_diff:
            print("✅ Sanity check PASSED: cùng speaker > khác speaker")
        else:
            print("⚠️  Sanity check: cùng speaker KHÔNG cao hơn khác speaker")
            print("   Điều này không nhất thiết sai (mỗi cặp file có thể khác nhau)")

    # Kiểm tra không có NaN/Inf
    all_embeddings = np.array([r["embedding"] for r in records])
    print()
    print(f"── Kiểm tra chất lượng toàn bộ embeddings ──")
    print(f"  Shape matrix   : {all_embeddings.shape}")
    print(f"  NaN count      : {np.isnan(all_embeddings).sum()}")
    print(f"  Inf count      : {np.isinf(all_embeddings).sum()}")
    print(f"  Global mean    : {all_embeddings.mean():.4f}")
    print(f"  Global std     : {all_embeddings.std():.4f}")

    if np.isnan(all_embeddings).sum() == 0 and np.isinf(all_embeddings).sum() == 0:
        print("\n✅ Không có NaN/Inf. Embeddings sạch, sẵn sàng đưa vào DB!")
    else:
        print("\n❌ Phát hiện NaN/Inf. Kiểm tra lại pipeline.")

✅ Cosine similarity (cùng speaker p225): 0.9950
   Cosine similarity (khác speaker p228): 0.9973

⚠️  Sanity check: cùng speaker KHÔNG cao hơn khác speaker
   Điều này không nhất thiết sai (mỗi cặp file có thể khác nhau)

── Kiểm tra chất lượng toàn bộ embeddings ──
  Shape matrix   : (1012, 99)
  NaN count      : 0
  Inf count      : 0
  Global mean    : 3.9870
  Global std     : 51.8409

✅ Không có NaN/Inf. Embeddings sạch, sẵn sàng đưa vào DB!
